In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
sys.path.append("/Users/Shivang/LLM from scratch/LLM-from-Scratch-Self/")
from Chapter_4.gpt import TansformersConfig

In [5]:
# LayerNorm
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        self.scale = torch.Parameter(torch.ones(emb_dim))
        self.bias = torch.Parameter(torch.zeros(emb_dim))
        self.epsilon = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdims=True)
        var = x.var(dim=-1, keepdims=True, unbiased=False)
        norm_x = x - mean / (torch.sqrt(self.epsilon + var))

        return norm_x * self.scale + self.bias


# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, epsilon=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(emb_dim))
        self.epsilon = epsilon

    def forward(self, x):
        means = torch.pow(x, 2).mean(dim=-1, keepdims=True)
        x_normed = x * torch.rsqrt(self.epsilon + means)
        return (x_normed * self.weight).to(dtype=x.dtype)


# checking if the implementation is working good
torch.manual_seed(123)

example_batch = torch.randn(2, 3, 4)

rms_norm = RMSNorm(emb_dim=example_batch.shape[-1])
rmsnorm_pytorch = torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

In [6]:
# GELU Implementation
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(seff, x):
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )


# SiLU Implementation
class SiLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)


# checking implementation
silu = SiLU()

assert torch.allclose(silu(example_batch), torch.nn.functional.silu(example_batch))

In [ ]:
# # GPT FeedForward
# class FeedForward(nn.Module):
#     def __init__(self, contants):
#         self.layers = nn.Sequential([nn.Linear(self.contants.contants.emb_dim, 4 * self.contants.emb_dim), 
#                     GELU(), 
#                     nn.Linear(4 * self.contants.emb_dim, self.contants.emb_dim)]
#                     )
#     def forward(self, x):
#         return self.layers(x)

## Llama 2 Feedforward
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.emb_dim, cfg.hidden_dim, dtype = cfg.dtype, bias= False)
        self.fc2 = nn.Linear(cfg.emb_dim, cfg.hidden_dim, dtype = cfg.dtype, bias= False)
        self.fc3 = nn.Linear(cfg.hidden_dim, cfg.emb_dim, dtype = cfg.dtype, bias = False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x_fc1)
        x = self.silu(x_fc1) * x_fc2
        return self.fc3(x)
